In [1]:
cd ..

c:\Users\Lenovo\Desktop\FYP\Tourist-Attraction-Recommendation-System


c:\Users\Lenovo\Desktop\FYP\Tourist-Attraction-Recommendation-System\venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


# Import Packages

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import random
import pandas as pd
from sklearn.model_selection import train_test_split




# Hyperparameters

In [4]:
lr = 0.001
batch_size = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# Dataset Preparation

In [33]:
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset

class NCFDataset(Dataset):
    def __init__(self, user, item, rating, negative_sampling=False):
        super(Dataset, self).__init__()
        
        # Encode user_id and place_id
        self.user_encoder = LabelEncoder()
        self.item_encoder = LabelEncoder()
        
        self.user = torch.LongTensor(self.user_encoder.fit_transform(user))
        self.item = torch.LongTensor(self.item_encoder.fit_transform(item))
        self.rating = torch.FloatTensor(rating.values)
        
        self.num_users = len(self.user_encoder.classes_)
        self.num_items = len(self.item_encoder.classes_)
        
        self.negative_sampling = negative_sampling
    
    def __len__(self):
        return len(self.user)
    
    def __getitem__(self, idx):
        user = self.user[idx]
        item = self.item[idx]
        rating = self.rating[idx]
        
        if self.negative_sampling and rating == 1:
            while True:
                negative_item = torch.randint(0, self.num_items, (1,)).item()
                if negative_item != item:
                    return user, item, rating, negative_item
        return user, item, rating


In [26]:
# Load datasets
attraction_df = pd.read_csv('./Data/FinalDataset/merged_all.csv')
ratings_df = pd.read_csv('./Data/FinalDataset/rating_final.csv')
ratings_df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3074 entries, 0 to 3073
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   user_id   3074 non-null   int64
 1   place_id  3074 non-null   int64
 2   rating    3074 non-null   int64
dtypes: int64(3)
memory usage: 72.2 KB


In [34]:

# Extract raw data
user = ratings_df['user_id']
item = ratings_df['place_id']
rating = ratings_df['rating']

# Train-test split
user_train, user_test, item_train, item_test, rating_train, rating_test = train_test_split(
    user, item, rating, test_size=0.2, random_state=42
)

# Create datasets
train_dataset = NCFDataset(user_train, item_train, rating_train, negative_sampling=True)
test_dataset = NCFDataset(user_test, item_test, rating_test, negative_sampling=False)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# Model Building

In [39]:
class GMF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim):
        super(GMF, self).__init__()
        self.user_embedding = nn.Embedding(num_users, latent_dim)   
        self.item_embedding = nn.Embedding(num_items, latent_dim)  
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight) 
    
    def forward(self, user, item):
        user_embedding = self.user_embedding(user)
        item_embedding = self.item_embedding(item)
        return user_embedding * item_embedding


In [37]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        
        